# Colab ViT-S/16 DINO: vit_s16_covidqu_syn

This notebook runs only `vit_s16_covidqu_syn`. It keeps pretraining and fine-tuning in separate cells so you can resume pretraining in epoch chunks.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone or Pull Repository

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/tlinhevg05/contrastive-synthesis-medcls_CVProject.git'
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')

if REPO_ROOT.exists():
    %cd {REPO_ROOT}
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_ROOT}

!git rev-parse --short HEAD

## 3. Install Minimal Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

packages = {
    'timm': 'timm',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'pandas': 'pandas',
    'PIL': 'Pillow',
}

to_install = []
for module_name, package_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        to_install.append(package_name)

if to_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *to_install])
else:
    print('All required packages already installed.')

import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi

## 4. Editable Paths and Run Flags

`PRETRAIN_EPOCH_OVERRIDE` is the total target DINO epoch count. If a `last_dino_checkpoint.pth` already exists in the output folder, pretraining resumes up to this total.

`LOCAL_CROPS_NUMBER=4` uses DINO multi-crop. If a `timm` version still rejects 96x96 crops, set it to `0` and rerun pretraining.


In [ ]:
REPO_ROOT = Path('/content/contrastive-synthesis-medcls_CVProject')
OUTPUT_ROOT = Path('/content/drive/MyDrive/medcls_cvproject/results/experiments')
SYNTHETIC_MANIFEST = Path('/content/drive/MyDrive/medcls_cvproject/data/manifests/synthetic_dcgan.csv')
REAL_UNLABELED_DIR = REPO_ROOT / 'data/processed/unlabelled_16934'

RUN_VIT_COVIDQU = False
RUN_VIT_IMAGENET_COVIDQU = False
RUN_VIT_COVIDQU_SYN = True
RUN_VIT_IMAGENET_COVIDQU_SYN = False

# Run pretraining in small chunks by increasing this total target: 10, 20, 30, ...
RUN_PRETRAIN = True
RUN_FINETUNE = False  # Set True only after the chosen pretraining target is finished.
PRETRAIN_EPOCH_OVERRIDE = 10
FINETUNE_EPOCH_OVERRIDE = None
LOCAL_CROPS_NUMBER = 4  # DINO default here. If timm crop-size errors persist, set this to 0.

pretrain_epoch_arg = '' if PRETRAIN_EPOCH_OVERRIDE is None else f'--epochs {PRETRAIN_EPOCH_OVERRIDE}'
local_crops_arg = f'--local-crops-number {LOCAL_CROPS_NUMBER}'
finetune_epoch_arg = '' if FINETUNE_EPOCH_OVERRIDE is None else f'--epochs {FINETUNE_EPOCH_OVERRIDE}'

%cd {REPO_ROOT}
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('SYNTHETIC_MANIFEST:', SYNTHETIC_MANIFEST, SYNTHETIC_MANIFEST.exists())
print('REAL_UNLABELED_DIR:', REAL_UNLABELED_DIR, REAL_UNLABELED_DIR.exists())
print('RUN_PRETRAIN:', RUN_PRETRAIN)
print('RUN_FINETUNE:', RUN_FINETUNE)
print('pretrain_epoch_arg:', pretrain_epoch_arg)
print('local_crops_arg:', local_crops_arg)
print('finetune_epoch_arg:', finetune_epoch_arg)

## 5. Verify Inputs and Scripts

In [ ]:
!python scripts/check_experiment_inputs.py --synthetic-manifest "{SYNTHETIC_MANIFEST}"
!python -m py_compile scripts/run_dino_vit.py scripts/run_classification_vit.py
!python scripts/run_dino_vit.py --help | grep resume || true
!python scripts/run_classification_vit.py --help | grep pretrained || true

## 6. Experiment: vit_s16_covidqu_syn

DINO pretraining from random initialization on Stage 1 DCGAN synthetic images, then supervised fine-tuning on real labeled manifests.

Run the pretraining cell repeatedly by increasing `PRETRAIN_EPOCH_OVERRIDE` from 10 to 20, 30, and so on. Run the fine-tuning cell only after pretraining reaches the target you want to report.

### 6a. Pretrain Only: vit_s16_covidqu_syn

In [ ]:
EXP = 'vit_s16_covidqu_syn'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_dino_teacher.pth'
RESUME_CKPT = OUT / 'pretrain/checkpoints/last_dino_checkpoint.pth'

if RUN_VIT_COVIDQU_SYN and RUN_PRETRAIN:
    !python scripts/run_dino_vit.py \
      --config configs/experiments/vit_s16/covidqu_syn.yaml \
      --synthetic-manifest "{SYNTHETIC_MANIFEST}" \
      --output-dir "{OUT}" \
      --resume-checkpoint "{RESUME_CKPT}" \
      {local_crops_arg} \
      {pretrain_epoch_arg}
else:
    print('Skipping pretrain', EXP)


### 6b. Fine-Tune Only: vit_s16_covidqu_syn

In [ ]:
EXP = 'vit_s16_covidqu_syn'
OUT = OUTPUT_ROOT / EXP
CKPT = OUT / 'pretrain/checkpoints/best_dino_teacher.pth'

if RUN_VIT_COVIDQU_SYN and RUN_FINETUNE:
    if not CKPT.exists():
        raise FileNotFoundError(f'DINO checkpoint not found: {CKPT}. Finish pretraining first.')
    !python scripts/run_classification_vit.py \
      --config configs/experiments/vit_s16/covidqu_syn.yaml \
      --manifest-dir data/manifests \
      --output-dir "{OUT}" \
      --pretrained-checkpoint "{CKPT}" \
      {finetune_epoch_arg}
else:
    print('Skipping finetune', EXP)


## 7. Display Result


In [ ]:
import json
import pandas as pd

EXP = 'vit_s16_covidqu_syn'
metrics_path = OUTPUT_ROOT / EXP / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    display(pd.DataFrame([{**{'experiment_id': EXP}, **metrics}]))
else:
    print('No metrics found yet:', metrics_path)

!find "{OUTPUT_ROOT}/vit_s16_covidqu_syn" -maxdepth 4 -type f \( -name 'metrics.json' -o -name 'best_dino_teacher.pth' -o -name 'last_dino_checkpoint.pth' -o -name 'confusion_matrix.png' -o -name 'classification_report.csv' \) | sort
